In [ ]:
# Colab dependency setup.
# Run this first in Google Colab, or use Runtime > Run all.
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    get_ipython().system(f"{sys.executable} -m pip install -q pandas numpy scikit-learn matplotlib openai")


# Notebook 3 — Outcome Evaluation and LLM-as-Judge Calibration
### Evaluating AI Agents: From Component Checks to Adversarial Defense

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Score agent outputs using a **multi-dimensional rubric** (factuality, completeness, groundedness, format, safety).
2. Run a **deterministic mock LLM-as-judge** to automate scoring at scale.
3. **Calibrate the judge** against human labels using Pearson/Spearman correlation, MAE, and agreement.
4. Identify which rubric dimensions the judge struggles with most — and why.

> **Why outcome evaluation is harder for agents?**  
> Agents produce open-ended answers with multiple quality dimensions.  
> A single scalar score misses whether the agent was *grounded*, *complete*, or *safe*.  
> LLM-as-judge scales rubric scoring — but only if the judge itself is calibrated against humans.

In [ ]:
#@title Setup and runtime options
import os
import sys
from pathlib import Path

#@markdown Check `USE_OPENAI` to call a small baseline OpenAI model. Add `OPENAI_API_KEY` in Colab Secrets first.
USE_OPENAI = False #@param {type:"boolean"}
OPENAI_MODEL = "gpt-4.1-nano" #@param {type:"string"}

REPO_URL = "https://github.com/AmmarMohanna/packt-ai-agents-eval.git"
REPO_NAME = "packt-ai-agents-eval"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_PATH = Path("/content") / REPO_NAME
    get_ipython().system(f"rm -rf {REPO_PATH}")
    get_ipython().system(f"git clone -q {REPO_URL} {REPO_PATH}")
    PROJECT_ROOT = REPO_PATH
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    PROJECT_ROOT = next(
        (path for path in candidates if (path / "src").is_dir() and (path / "data").is_dir()),
        cwd,
    )

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT / "notebooks")

import importlib
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        sys.modules.pop(module_name, None)

OPENAI_API_KEY = None
if USE_OPENAI:
    if not IN_COLAB:
        raise RuntimeError("OpenAI mode is configured for Google Colab Secrets.")
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise RuntimeError("Add OPENAI_API_KEY in the Colab Secrets panel, then rerun the notebook.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110

from src.trace_utils import load_jsonl
from src.judge import mock_llm_judge
from src.openai_agents import judge_outcome_with_openai
from src.metrics import (
    pearson_corr, spearman_corr, mean_absolute_error,
    agreement_within_one, dimension_error_report, DIMENSIONS,
)

# OpenAI mode is controlled by the USE_OPENAI checkbox at the top of this cell.

pd.set_option("display.max_colwidth", 65)
print("Setup complete.")

In [ ]:
# ── Load outcome dataset and human labels ─────────────────────────────────
DATA_PATH   = os.path.join("..", "data", "outcome_dataset.jsonl")
LABELS_PATH = os.path.join("..", "data", "human_labels.csv")

examples = load_jsonl(DATA_PATH)
human_df = pd.read_csv(LABELS_PATH)

print(f"Outcome examples:  {len(examples)}")
print(f"Human label rows:  {len(human_df)}")
print("\nHuman labels sample:")
print(human_df.to_string(index=False))

## Rubric Dimensions

Each agent answer is scored 1–5 on six dimensions:

| Dimension | What it measures |
|-----------|------------------|
| **factuality** | Are the claims in the answer accurate relative to the evidence? |
| **completeness** | Does the answer cover all key points the evidence supports? |
| **groundedness** | Is every claim traceable to a specific piece of evidence? |
| **format_adherence** | Does the answer follow any formatting instructions (bullets, citations, etc.)? |
| **safety** | Does the answer avoid harmful, misleading, or overconfident claims? |
| **overall** | Holistic quality across all dimensions. |

**Score scale:** 1 = very poor, 3 = acceptable, 5 = excellent.

In [ ]:
# ── Run the LLM-as-judge on all examples ──────────────────────────────────
#
# By default we use the deterministic mock judge (USE_OPENAI = False).
# The mock judge uses heuristic rules — it intentionally makes some mistakes
# to give us meaningful calibration signal.
# Optional OpenAI mode reads OPENAI_API_KEY from Colab Secrets.

if USE_OPENAI:
    judge_scores = [judge_outcome_with_openai(ex, api_key=OPENAI_API_KEY, model=OPENAI_MODEL) for ex in examples]
else:
    judge_scores = [mock_llm_judge(ex) for ex in examples]

judge_df = pd.DataFrame(judge_scores)

print(f"Judge scores generated for {len(judge_df)} examples.")
print("\nJudge scores:")
print(judge_df.to_string(index=False))

In [ ]:
# ── Merge judge scores with human labels ──────────────────────────────────
merged = human_df.merge(judge_df, on="id", suffixes=("_human", "_judge"))

print(f"Merged {len(merged)} rows.")
print("\nSide-by-side comparison (human vs judge) — 'factuality' column:")
cols = ["id"] + [f"{d}_human" for d in DIMENSIONS] + [f"{d}_judge" for d in DIMENSIONS]
available = [c for c in cols if c in merged.columns]
print(merged[available].to_string(index=False))

In [ ]:
# ── Compute per-dimension calibration metrics ──────────────────────────────
human_scores = {dim: merged[f"{dim}_human"].tolist() for dim in DIMENSIONS if f"{dim}_human" in merged.columns}
judge_scores_dict = {dim: merged[f"{dim}_judge"].tolist() for dim in DIMENSIONS if f"{dim}_judge" in merged.columns}

report = dimension_error_report(human_scores, judge_scores_dict)
df_report = pd.DataFrame(report)

print("Dimension-level calibration report:")
print(df_report.to_string(index=False))

In [ ]:
# ── Visualise calibration: heatmap of metrics by dimension ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: Pearson and Spearman
corr_data = df_report.set_index("dimension")[["pearson_r", "spearman_r"]]
im0 = axes[0].imshow(corr_data.values, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Pearson r", "Spearman r"])
axes[0].set_yticks(range(len(corr_data))); axes[0].set_yticklabels(corr_data.index)
axes[0].set_title("Correlation with Human Labels")
for i in range(len(corr_data)):
    for j in range(2):
        axes[0].text(j, i, f"{corr_data.values[i, j]:.2f}", ha="center", va="center", fontsize=10)
plt.colorbar(im0, ax=axes[0])

# Right: MAE and agreement
err_data = df_report.set_index("dimension")[["mae", "agreement_within_1"]]
im1 = axes[1].imshow(err_data.values, cmap="RdYlGn_r", vmin=0, vmax=2, aspect="auto")
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["MAE", "Agreement±1"])
axes[1].set_yticks(range(len(err_data))); axes[1].set_yticklabels(err_data.index)
axes[1].set_title("Error and Agreement")
for i in range(len(err_data)):
    for j in range(2):
        axes[1].text(j, i, f"{err_data.values[i, j]:.2f}", ha="center", va="center", fontsize=10)
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "nb03_judge_calibration.png"), bbox_inches="tight")
plt.show()

In [ ]:
# ── Find examples where judge disagrees most with humans ──────────────────
# Compute per-row, per-dimension absolute error
for dim in DIMENSIONS:
    if f"{dim}_human" in merged.columns and f"{dim}_judge" in merged.columns:
        merged[f"{dim}_err"] = (merged[f"{dim}_human"] - merged[f"{dim}_judge"]).abs()

err_cols = [c for c in merged.columns if c.endswith("_err")]
merged["max_err"] = merged[err_cols].max(axis=1)
merged["worst_dim"] = merged[err_cols].idxmax(axis=1).str.replace("_err", "")

worst = merged.sort_values("max_err", ascending=False).head(4)
print("Examples where judge disagreed most with human labels:")
print(worst[["id", "worst_dim", "max_err"] +
            [f"{d}_human" for d in DIMENSIONS if f"{d}_human" in merged.columns] +
            [f"{d}_judge" for d in DIMENSIONS if f"{d}_judge" in merged.columns]
           ].to_string(index=False))

## Practical Judge Calibration Lessons

After running this calibration, look for these patterns:

| Pattern | Root cause | Remedy |
|---------|------------|--------|
| **Low Pearson r on groundedness** | Judge can't detect missing citations | Add explicit citation presence check to rubric prompt |
| **High MAE on safety** | Judge under-scores obvious safety issues | Include example unsafe answers as negative few-shot examples |
| **High agreement on factuality** | Judge benefits from same evidence text | No fix needed — but verify on out-of-distribution examples |
| **Judge inflates completeness for long answers** | Length bias | Penalise verbosity; check whether all evidence sources are covered |

**Key insight:** The mock judge in this notebook intentionally over-rewards verbosity (completeness) and under-detects missing citations (groundedness). These are common real-world LLM judge failure modes.

**Before trusting a judge in production:**
- Collect at least 50–100 human labels across your actual task distribution.
- Target Pearson r > 0.7 and agreement-within-1 > 80% on your most safety-critical dimensions.
- Re-calibrate after every major model or prompt change.

## Try It Yourself

**Extension exercise** — Modify one judge rule and rerun calibration:

In `src/judge.py`, find `_score_completeness()` and change the length reward:

```python
# Current (biased — rewards long answers):
if len(answer) > 300:
    return 4  # intentional bias

# Try this fix instead:
# Remove the length reward and only check evidence coverage:
if len(answer) > 300:
    return 3  # neutral — length alone is not quality
```

Then re-run cells 5–7 in this notebook and observe:
- Does the MAE for `completeness` improve?
- Does fixing one dimension affect other dimensions?

**Questions to explore:**
- Which dimension would you prioritise for the research assistant use case — safety or completeness?
- How many human labels would you need to get a statistically reliable calibration estimate?